# 랜덤신호론 15주차 실습
## 최종 프로젝트와 모델 해석

1. 최종 모델을 처음부터 끝까지 실행한다.
2. 새 신호를 분석하고 근거를 설명한다.
3. 혼동행렬과 오분류를 해석한다.
4. 모델의 한계와 개선 방향을 제안한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. 최종 모델 실행


In [ ]:
fs=50;df=build_dataset(70,fs);X=df[FEATURE_NAMES];y=df.label
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
model=make_pipeline(StandardScaler(),KNeighborsClassifier(5));model.fit(X_train,y_train);y_pred=model.predict(X_test)
print('정확도:',accuracy_score(y_test,y_pred));print(classification_report(y_test,y_pred,zero_division=0))


## 2. 혼동행렬과 오분류


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap='Blues');plt.show()
result=X_test.copy();result['actual']=y_test.to_numpy();result['predicted']=y_pred
errors=result[result.actual!=result.predicted];print('오분류:',len(errors));print(errors.head())


**관찰 1** 가장 많이 혼동한 상태와 특징을 설명하세요.


## 3. 새로운 신호 분석


In [ ]:
def analyze(signal):
    f=extract_features(signal,fs);xnew=pd.DataFrame([[f[n] for n in FEATURE_NAMES]],columns=FEATURE_NAMES)
    return model.predict(xnew)[0],f
for i,k in enumerate(['normal','high_noise','spike']):
    _,sig=generate_signal(k,fs,seed=900+i);pred,f=analyze(sig)
    print('실제',k,'예측',pred,'std',round(f['std'],2),'p2p',round(f['peak_to_peak'],2))


## 4. 결과 저장


In [ ]:
summary=pd.DataFrame({'metric':['accuracy','train','test'],'value':[accuracy_score(y_test,y_pred),len(X_train),len(X_test)]})
print(summary);summary.to_csv('week15_model_summary.csv',index=False,encoding='utf-8-sig')


### 최종 체크리스트
- 모든 셀을 순서대로 실행했습니다.
- 특징과 모델 설정을 설명했습니다.
- 혼동행렬과 오분류를 해석했습니다.
- 실제 적용 한계를 3가지 이상 작성했습니다.


## 참고 자료
- classification_report: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
- ConfusionMatrixDisplay: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html
- UCI Sensorless Drive Diagnosis: https://archive.ics.uci.edu/dataset/325/dataset+for+sensorless+drive+diagnosis
